# Rare Disease Case Report to HPO-Coded Patient Record

This is catchfly's flagship example. No other open-source library takes a raw clinical case report and produces a structured patient record with standardized ontology codes in a single pipeline.

**What you'll build:** A pipeline that reads rare disease case reports and outputs structured records where every phenotype is mapped to the Human Phenotype Ontology (HPO) with confidence scores.

**Estimated API cost:** ~$0.05

In [ ]:
# pip install "catchfly[openai,medical,export,chunking]"
# export OPENAI_API_KEY="sk-..."

## Why this matters

Researchers building rare disease patient cohorts from published case reports spend weeks on:

1. Reading each report and deciding what fields to extract
2. Manually copying values into spreadsheets
3. Normalizing inconsistent terminology ("seizures" vs "epileptic fits" vs "convulsions")
4. Looking up HPO codes for each phenotype

Catchfly automates all four steps.

## Step 1: Load case reports

Catchfly bundles 10 rare disease case reports (Niemann-Pick, Fabry, Hunter syndrome, Pompe, Gaucher, Morquio, Cystinosis, Tay-Sachs):

In [ ]:
from catchfly.demo import load_samples

docs = load_samples("case_reports")
print(f"{len(docs)} case reports loaded")
print(f"\nFirst report preview ({len(docs[0].content)} chars):")
print(docs[0].content[:300] + "...")

## Step 2: Discover the schema

Instead of hand-crafting a Pydantic model, let the LLM figure out what fields exist across your corpus. `ThreeStageDiscovery` progressively refines the schema across 3 stages with increasing sample sizes.

In [ ]:
from catchfly.discovery import ThreeStageDiscovery

discovery = ThreeStageDiscovery(
    model="gpt-5.4-mini",
    stage1_samples=3,
    stage2_samples=5,
    stage3_samples=10,
)
schema = discovery.discover(docs, domain_hint="Rare disease clinical case reports")

print("Discovered fields:")
for name, prop in schema.json_schema["properties"].items():
    print(f"  {name}: {prop.get('type', 'object')}")

## Step 3: Enrich the schema

`SchemaOptimizer` runs extraction on test documents and asks the LLM to improve field descriptions based on errors and gaps. These enriched descriptions improve both extraction quality and downstream normalization.

In [ ]:
from catchfly.discovery import SchemaOptimizer

optimizer = SchemaOptimizer(model="gpt-5.4-mini", num_iterations=2)
enriched = optimizer.optimize(schema, test_documents=docs[:5])

# Inspect enriched field metadata
meta = enriched.field_metadata.get("phenotypes", {})
print(f"Description: {meta.get('description', '(none)')}")
print(f"Examples: {meta.get('examples', [])}")
print(f"Synonyms: {meta.get('synonyms', [])}")

## Step 4: Extract structured records

With `SentenceChunking` (via chonkie), long case reports are split at sentence boundaries instead of mid-word.

In [ ]:
from catchfly.extraction import LLMDirectExtraction, SentenceChunking

extractor = LLMDirectExtraction(
    model="gpt-5.4-mini",
    chunking_strategy=SentenceChunking(chunk_size=512, overlap=64),
    on_error="collect",
)

result = extractor.extract(enriched.model, docs)

print(f"Extracted {len(result.records)} records, {len(result.errors)} errors\n")
for record in result.records[:3]:
    print(f"  {record.diagnosis}: {getattr(record, 'phenotypes', [])}")

### Provenance tracking

Every extracted value traces back to the exact character offset in the source document:

In [ ]:
prov = result.provenance[0]
print(f"Source: {prov.source_document}")
print(f"Chars: {prov.char_start}-{prov.char_end}")
print(f"Confidence: {prov.confidence}")

## Step 5: Normalize with cascading strategies

This is where catchfly truly differentiates. `CascadeNormalization` chains three strategies:

1. **Dictionary** — known medical abbreviations (instant, zero cost)
2. **LLM** — semantic grouping of synonyms
3. **Ontology** — map to HPO codes with confidence scores

In [ ]:
from catchfly.normalization import CascadeNormalization

cascade = CascadeNormalization.default(
    dictionary={
        "ALT": "Alanine aminotransferase",
        "AST": "Aspartate aminotransferase",
        "CK": "Creatine kinase",
        "GFR": "Glomerular filtration rate",
        "GAGs": "Glycosaminoglycans",
    },
    model="gpt-5.4-mini",
    ontology="hpo",
)

# Collect all phenotype values across records
all_phenotypes = []
for record in result.records:
    if hasattr(record, "phenotypes") and record.phenotypes:
        all_phenotypes.extend(record.phenotypes)

print(f"Normalizing {len(all_phenotypes)} phenotype values ({len(set(all_phenotypes))} unique)")
norm_result = cascade.normalize(all_phenotypes, context_field="phenotypes")

### What each cascade step accomplished

In [ ]:
for step in norm_result.metadata["steps"]:
    print(f"Step {step['step']} ({step['strategy']}): "
          f"mapped {step['mapped_count']}, remaining {step['remaining_count']}")

### HPO codes with confidence scores

In [ ]:
# Show sample mappings with ontology IDs
sample_values = ["seizures", "hepatosplenomegaly", "corneal verticillata",
                 "muscle weakness", "hearing loss"]

for value in sample_values:
    canonical = norm_result.mapping.get(value, value)
    explanation = norm_result.explain(value)
    print(f"  {explanation}")

### All normalized clusters

In [ ]:
print(f"{norm_result.metadata['n_groups']} canonical groups:\n")
for canonical, members in sorted(norm_result.clusters.items()):
    if len(members) > 1:
        print(f"  {canonical}: {members}")

## Full pipeline (all-in-one)

For production use, the `Pipeline` class orchestrates everything in a single call.

Here we use explicit `normalize_fields` because we're routing specific fields to a domain-specific `CascadeNormalization` with HPO ontology. For general-purpose normalization, `Pipeline.quick()` auto-selects fields via `StatisticalFieldSelector` — no `normalize_fields` needed.

In [ ]:
from catchfly import Pipeline
from catchfly.discovery import ThreeStageDiscovery, SchemaOptimizer
from catchfly.extraction import LLMDirectExtraction, SentenceChunking
from catchfly.normalization import CascadeNormalization
from catchfly.demo import load_samples

docs = load_samples("case_reports")

pipeline = Pipeline(
    discovery=ThreeStageDiscovery(model="gpt-5.4-mini"),
    extraction=LLMDirectExtraction(
        model="gpt-5.4-mini",
        chunking_strategy=SentenceChunking(chunk_size=512),
        on_error="collect",
    ),
    normalization=CascadeNormalization.default(
        dictionary={"ALT": "Alanine aminotransferase", "AST": "Aspartate aminotransferase"},
        model="gpt-5.4-mini",
        ontology="hpo",
    ),
)

results = pipeline.run(
    docs,
    domain_hint="Rare disease clinical case reports",
    normalize_fields=["phenotypes", "diagnosis"],
    max_cost_usd=1.0,
)

print(f"Records: {len(results.records)}")
print(f"Cost: ${results.report.total_cost_usd:.3f}")
results.to_dataframe()

## Production tips

- **Real data:** Replace `load_samples()` with PMC full-text articles via `load_glob("data/case_reports/*.txt")`
- **Custom ontology:** `OntologyMapping(ontology="path/to/custom_terms.csv")`
- **Checkpoint/resume:** Add `checkpoint_dir="./checkpoints"` for large corpora
- **Schema review:** Use `on_schema_ready` callback to inspect the discovered schema before extraction
- **Cost budget:** Set `max_cost_usd=5.0` to prevent runaway spending